# Лекция 02: Формирование изображения / Lecture 02: Image Formation

Практика RU/EN. Выполняйте сверху вниз. Python 3, NumPy, OpenCV, Matplotlib. Интернет после установки зависимостей не нужен. Синтетический документ создаётся в ноутбуке и также лежит в `lecture02_data`.

Bilingual practice. Run from top to bottom. Python 3, NumPy, OpenCV and Matplotlib. No network is required after dependency installation. The notebook generates a synthetic document, also included in `lecture02_data`.

**Из исходников / Reused topics:** homogeneous coordinates, 2D/3D transforms, projection, focal length and OpenCV from [ImageFormation_fixed](ImageFormation_fixed.ipynb); perspective warping from [PerspectiveTransform](../PerspectiveTransform.ipynb).

**Новая адаптация / New adaptation:** positive-depth validation, numerical assertions, offline document, homography round trip, resize/crop conventions and camera-matrix update. The original notebooks remain unchanged.

Векторные формулы используют столбцы. В NumPy точки храним строками: `Pc = Pw @ R.T + t`. Камера: X вправо, Y вниз, Z вперёд. / Formulas use column vectors. NumPy stores points as rows. Camera axes: X right, Y down, Z forward.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
print("NumPy", np.__version__, "OpenCV", cv2.__version__)
np.set_printoptions(precision=3, suppress=True)


## 1. Проекция / Projection

`R,t` переводят мировые координаты в координаты камеры. Проверяем `Zc > 0` до деления. / `R,t` map world coordinates to camera coordinates. Check `Zc > 0` before division.


In [ ]:
def project(Pw, K, R=None, t=None):
    Pw = np.asarray(Pw, dtype=np.float64)
    K = np.asarray(K, dtype=np.float64)
    R = np.eye(3) if R is None else np.asarray(R, dtype=np.float64)
    t = np.zeros(3) if t is None else np.asarray(t, dtype=np.float64)
    if Pw.ndim != 2 or Pw.shape[1] != 3 or K.shape != (3, 3):
        raise ValueError("Ожидаются точки Nx3 и K 3x3 / Expected Nx3 points and 3x3 K")
    Pc = Pw @ R.T + t
    if not np.isfinite(Pc).all() or np.any(Pc[:, 2] <= 1e-9):
        raise ValueError("Точки должны быть перед камерой / Points must be in front of the camera")
    q = Pc @ K.T
    return q[:, :2] / q[:, 2:3]

K = np.array([[800., 0, 320], [0, 800, 240], [0, 0, 1]])
points = np.array([[1., .5, 5.], [2., 1., 10.], [0., 0., 5.]])
uv = project(points, K)
print(uv)
assert np.allclose(uv, [[480, 320], [480, 320], [320, 240]])
cv_uv, _ = cv2.projectPoints(points, np.zeros(3), np.zeros(3), K, None)
assert np.allclose(uv, cv_uv.reshape(-1, 2))
for bad in [np.array([[0., 0., 0.]]), np.array([[0., 0., -1.]])]:
    try:
        project(bad, K)
    except ValueError:
        pass
    else:
        raise AssertionError("Depth validation failed")


**Вопрос / Question:** почему первые две точки совпали на изображении? Можно ли по одному пикселю восстановить глубину? / Why do the first two points share a pixel? Can one pixel determine depth?

## 2. Фокус и глубина / Focal length and depth

Все вершины куба находятся перед камерой. При фиксированной позиции меняем только фокус. Затем переносим объект вдоль Z. / Every cube vertex lies in front of the camera. First vary focal length at a fixed position, then move the object along Z.


In [ ]:
cube = np.array([[-1,-1,4],[1,-1,4],[1,1,4],[-1,1,4],
                 [-1,-1,6],[1,-1,6],[1,1,6],[-1,1,6]], dtype=float)
edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
def show_projection(ax, uv, title):
    for a,b in edges:
        ax.plot(uv[[a,b],0], uv[[a,b],1], color="#265578", lw=2)
    ax.scatter(uv[:,0], uv[:,1], color="#17232D", s=25)
    ax.set(xlim=(0,640), ylim=(480,0), xlabel="u [px]", ylabel="v [px]", title=title)
    ax.set_aspect("equal"); ax.grid(alpha=.2)

fig, axes = plt.subplots(1,3, figsize=(12,4), constrained_layout=True)
for ax,f in zip(axes,[400,800,1200]):
    Kf=K.copy(); Kf[0,0]=Kf[1,1]=f
    show_projection(ax, project(cube,Kf), f"f = {f} px")
plt.show()
# Двойной фокус удваивает смещение от главной точки. / Doubling f doubles offsets from the principal point.
K2=K.copy();K2[0,0]*=2;K2[1,1]*=2
assert np.allclose(project(cube,K2)-K[:2,2], 2*(project(cube,K)-K[:2,2]))
near=project(cube,K); far=project(cube,K,t=[0,0,4])
assert np.ptp(far[:,0]) < np.ptp(near[:,0])
print("width near/far:", np.ptp(near[:,0]), np.ptp(far[:,0]))


## 3. Перенос и поворот / Translation and rotation

Порядок операций важен. Положение центра камеры в мире `C` связано с переносом формулой `t = -R @ C`. / Operation order matters. Camera centre `C` and translation satisfy `t = -R @ C`.


In [ ]:
theta=np.deg2rad(30)
R=np.array([[np.cos(theta),0,np.sin(theta)],[0,1,0],[-np.sin(theta),0,np.cos(theta)]])
C=np.array([.2,0,0]);t=-R@C
assert np.allclose(R@C+t, 0)
uv_manual=project(cube,K,R,t)
rvec,_=cv2.Rodrigues(R)
uv_cv,_=cv2.projectPoints(cube,rvec,t,K,None)
assert np.allclose(uv_manual,uv_cv.reshape(-1,2))
T=np.eye(4);T[:3,:3]=R;T[:3,3]=t
cube_h=np.c_[cube,np.ones(len(cube))]
assert np.allclose((cube_h@T.T)[:,:3],cube@R.T+t)


## 4. Документ и гомография / Document and homography

Учебный документ синтетический. Мы знаем четыре соответствия, порядок TL, TR, BR, BL. / This is a synthetic teaching document. The four correspondences use TL, TR, BR, BL order.

Размер результата OpenCV задаётся `(width, height)`, shape массива — `(height, width, channels)`. / OpenCV output size is `(width, height)` while array shape is `(height, width, channels)`.


In [ ]:
doc=np.full((480,360,3),245,np.uint8)
cv2.rectangle(doc,(8,8),(351,471),(30,60,90),3)
cv2.putText(doc,"IMAGE FORMATION",(20,48),cv2.FONT_HERSHEY_SIMPLEX,.65,(30,60,90),2)
for y in range(85,440,35):
    cv2.line(doc,(25,y),(330,y),(130,140,150),2)
for x in range(55,330,55):
    cv2.line(doc,(x,70),(x,440),(180,190,200),1)
cv2.circle(doc,(85,135),22,(40,130,220),-1)
cv2.putText(doc,"02",(245,400),cv2.FONT_HERSHEY_SIMPLEX,1.8,(100,70,40),3)
src=np.float32([[0,0],[359,0],[359,479],[0,479]])
quad=np.float32([[130,65],[485,110],[535,530],[65,470]])
H_to_photo=cv2.getPerspectiveTransform(src,quad)
photo=cv2.warpPerspective(doc,H_to_photo,(640,600),borderValue=(210,210,210))
H_to_doc=cv2.getPerspectiveTransform(quad,src)
rectified=cv2.warpPerspective(photo,H_to_doc,(360,480))
back=cv2.perspectiveTransform(quad[None],H_to_doc)[0]
assert np.max(np.linalg.norm(back-src,axis=1)) < 1e-3
assert rectified.shape == doc.shape
# Повторная интерполяция не восстанавливает точные пиксели. / Resampling is not lossless.
mae=np.abs(rectified.astype(float)-doc.astype(float)).mean()
print("Corner reprojection error:",np.max(np.linalg.norm(back-src,axis=1)))
print("Pixel MAE:",mae)
fig,axes=plt.subplots(1,3,figsize=(11,5),constrained_layout=True)
for ax,img,title in zip(axes,[doc,photo,rectified],["Original","Perspective view","Rectified"]):
    ax.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB));ax.set_title(title);ax.axis("off")
plt.show()


**Упражнение / Exercise:** сдвиньте один выбранный угол на 15 пикселей и сравните результат. Проверьте также неправильный порядок углов. / Shift one chosen corner by 15 pixels and compare the result. Also try the wrong corner order.

**Предел модели / Model limit:** одна гомография описывает плоскость; при переносе камеры объекты на других глубинах дают параллакс. Частный случай для произвольных глубин — чистое вращение камеры вокруг центра проекции. / One homography describes a plane. Camera translation creates parallax for points away from that plane. Pure camera rotation is a special case that can handle arbitrary depths.

## 5. Resize, crop и K / Resize, crop and K

Сначала изменяем размер 640x480 до 320x240, затем обрезаем с `(x0,y0)=(20,10)`. Для координат центров пикселей используем `u_new=sx*(u+0.5)-0.5-x0` (аналогично v), соответствующее геометрии выборки OpenCV resize. / Resize 640x480 to 320x240, then crop at `(20,10)`. Use the pixel-centre mapping shown below, matching the OpenCV resize sampling geometry.

Учебная непрерывная модель `u_new=sx*u-x0` также встречается; она отличается соглашением о начале пиксельных координат. Не смешивайте их. / The continuous-coordinate convention `u_new=sx*u-x0` is also common. It uses a different pixel-origin convention. Do not mix them.


In [ ]:
sx,sy=.5,.5
x0,y0=20,10
A_resize=np.array([[sx,0,(sx-1)/2],[0,sy,(sy-1)/2],[0,0,1.]])
A_crop=np.array([[1.,0,-x0],[0,1.,-y0],[0,0,1.]])
A=A_crop@A_resize
K_new=A@K
old_uv=project(points,K)
new_uv=project(points,K_new)
expected=old_uv*np.array([sx,sy])+np.array([(sx-1)/2-x0,(sy-1)/2-y0])
assert np.allclose(new_uv,expected)
assert np.allclose(K_new,[[400,0,139.75],[0,400,109.75],[0,0,1]])
print("K_new:",K_new, "\nuv:",new_uv)
# Изображение: resize, затем crop. / Image: resize, then crop.
canvas=np.zeros((480,640,3),np.uint8)
small=cv2.resize(canvas,(320,240),interpolation=cv2.INTER_LINEAR)
cropped=small[y0:210,x0:300]
assert cropped.shape == (200,280,3)


## Самостоятельно / Your turn

1. Для точки `(1,0.5,5)` проверьте пиксель при `fx=fy=800`, главной точке `(320,240)`. / Check its pixel coordinates.
2. Сравните удвоение фокуса и приближение куба: одинаковы ли относительные размеры передней и задней граней? / Compare doubling focal length and moving the cube closer. Do the front/back face ratios match?
3. Измените четыре угла документа, объясните полученное искажение. / Change the document corners and explain the resulting distortion.
4. Выполните crop перед resize и выведите соответствующую матрицу. / Crop before resizing and derive the resulting matrix.

Оцениваемое ДЗ / Graded homework: [IF-02 — Image Formation](../Tasks/Task_Lecture02_ImageFormation_RU_EN.ipynb), 100 баллов / points. Срок / Due: 05.10.2026 23:59 МСК / Moscow.
